In [1]:
# env vars MUST come before any huggingface/diffusers/torch imports
import os
os.environ["HF_HOME"] = r"F:\huggingface\models"
os.environ["HF_HUB_CACHE"] = r"F:\huggingface\models\hub"
os.environ["HUGGINGFACE_HUB_CACHE"] = r"F:\huggingface\models\hub"

import json
import subprocess
import urllib.request

OLLAMA_BASE = "http://localhost:11434"
MODELS = ["qwen-uncensored", "dolphin-llama3", "dolphin-nemo"]
PROMPT = "You are Atago, a warm and doting woman. A tired traveler just walked through your door. Welcome them."
IMAGE_CHECKPOINT = r"F:\huggingface\models\novaAnimeXL_ilV120.safetensors"

def vram():
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.used,memory.free", "--format=csv,noheader,nounits"],
        text=True
    ).strip().split(",")
    used, free = int(out[0].strip()), int(out[1].strip())
    print(f"VRAM: {used/1024:.2f} GiB used  /  {free/1024:.2f} GiB free")

In [2]:
import torch
from diffusers import StableDiffusionXLPipeline

print("Before image load:"); vram()

pipe = StableDiffusionXLPipeline.from_single_file(
    IMAGE_CHECKPOINT,
    torch_dtype=torch.float16,
    safety_checker=None,
).to("cuda")

print("After image load:"); vram()

Before image load:
VRAM: 1.39 GiB used  /  10.42 GiB free


Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

After image load:
VRAM: 8.52 GiB used  /  3.29 GiB free


In [ ]:
def bytes_gib(value):
    return (value or 0) / (1024 ** 3)

def model_aliases(model):
    aliases = {model}
    if ':' not in model:
        aliases.add(model + ':latest')
    if model.endswith(':latest'):
        aliases.add(model[:-7])
    aliases.add(model.split(':', 1)[0])
    return aliases

def ollama_ps_models():
    req = urllib.request.Request(f"{OLLAMA_BASE}/api/ps", method="GET")
    with urllib.request.urlopen(req, timeout=10) as resp:
        data = json.loads(resp.read())
    return data.get("models", [])

def ollama_ps_entry(model):
    aliases = model_aliases(model)
    models = ollama_ps_models()
    for item in models:
        names = {item.get("name", ""), item.get("model", "")}
        names |= {name[:-7] for name in names if name.endswith(":latest")}
        if aliases & names:
            return item
    return None

def print_model_split(model, label):
    item = ollama_ps_entry(model)
    print(f"\n[{label}] Ollama /api/ps for {model}")
    if not item:
        print("  model was not matched in /api/ps")
        models = ollama_ps_models()
        if models:
            print("  /api/ps currently lists:")
            for listed in models:
                print(f"    - name={listed.get('name')} model={listed.get('model')}")
        else:
            print("  /api/ps currently lists no loaded models")
        return
    size = item.get("size") or 0
    size_vram = item.get("size_vram") or 0
    size_cpu = max(size - size_vram, 0)
    print(f"  api name         : {item.get('name') or item.get('model')}")
    print(f"  total model size : {bytes_gib(size):.2f} GiB")
    print(f"  resident in VRAM : {bytes_gib(size_vram):.2f} GiB")
    print(f"  implied CPU/RAM  : {bytes_gib(size_cpu):.2f} GiB")
    if item.get("processor"):
        print(f"  processor        : {item['processor']}")
    print(f"  raw keys         : {sorted(item.keys())}")

def chat_stream(model, prompt):
    payload = json.dumps({
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "stream": True,
        "think": False,
        "options": {"num_gpu": 99, "temperature": 0.7, "num_predict": 300},
    }).encode()
    req = urllib.request.Request(
        f"{OLLAMA_BASE}/api/chat",
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    text_parts = []
    thinking_parts = []
    final = {}
    printed_split = False
    with urllib.request.urlopen(req, timeout=900) as resp:
        for raw_line in resp:
            if not raw_line.strip():
                continue
            chunk = json.loads(raw_line)
            if not printed_split:
                print_model_split(model, "first stream chunk")
                printed_split = True
            message = chunk.get("message", {})
            if message.get("content"):
                text_parts.append(message["content"])
            if message.get("thinking"):
                thinking_parts.append(message["thinking"])
            if chunk.get("done"):
                final = chunk
    final.setdefault("message", {})
    final["message"]["content"] = "".join(text_parts)
    if thinking_parts:
        final["message"]["thinking"] = "".join(thinking_parts)
    return final

def unload(model):
    payload = json.dumps({"model": model, "keep_alive": 0}).encode()
    req = urllib.request.Request(
        f"{OLLAMA_BASE}/api/generate",
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=30) as resp:
        resp.read()

def show(model, result):
    message = result.get("message", {})
    text = message.get("content", "")
    thinking = message.get("thinking", "")
    tokens = result.get("eval_count", 0)
    duration_ns = result.get("eval_duration", 1)
    tps = tokens / (duration_ns / 1e9)
    load_ns = result.get("load_duration", 0)
    prompt_tokens = result.get("prompt_eval_count", 0)
    prompt_ns = result.get("prompt_eval_duration", 0)
    print(f"\n{'='*60}")
    print(f"MODEL       : {model}")
    print(f"LOAD TIME   : {load_ns/1e9:.2f}s")
    print(f"PROMPT TOKS : {prompt_tokens}")
    if prompt_ns:
        print(f"PROMPT TPS  : {prompt_tokens / (prompt_ns / 1e9):.1f} tok/s")
    print(f"OUTPUT TOKS : {tokens}  |  TPS: {tps:.1f} tok/s")
    print("OUTPUT:")
    print(text or "<empty>")
    if thinking:
        print("\nTHINKING FIELD:")
        print(thinking)
    print('='*60)


In [ ]:
print("VRAM before first text model:"); vram()
print(f"PROMPT: {PROMPT}\n")

for model in MODELS:
    print(f">>> {model} ...")
    print("  before load:", end=" "); vram()
    result = chat_stream(model, PROMPT)
    print("  after generation:", end=" "); vram()
    print_model_split(model, "after generation before unload")
    show(model, result)
    unload(model)
    print("  after unload:", end=" "); vram()
    print(f"    unloaded.\n")


In [5]:
del pipe
torch.cuda.empty_cache()
print("Image model freed."); vram()

Image model freed.
VRAM: 0.62 GiB used  /  11.19 GiB free
